# Decision tree comparison: from scratch vs scikit-learn

This notebook is a reproducible, side-by-side validation of the project's NumPy decision tree and `sklearn.tree.DecisionTreeClassifier`. It compares both models on the same Iris data, with the same train/test split, Gini criterion, and depth limits. The goal is to understand agreement and implementation differences, not to claim that the educational implementation is a drop-in replacement.

## Experimental design

- **Data:** Iris, 150 rows, four numeric features, and three classes.
- **Split:** one fixed 80/20 stratified split (`random_state=42`), shared by both models.
- **Criterion:** Gini impurity for both implementations.
- **Depth:** the custom tree and scikit-learn tree use the same `max_depth` in each experiment.
- **Preprocessing:** no scaling. Tree thresholds are invariant to monotonic feature scaling, so scaling would add noise without improving this comparison.
- **Metrics:** accuracy, exact sample-level prediction agreement, confusion matrices, depth, and node count.

The custom model's `min_leaf_size` is a node-level split guard. It is therefore not identical to scikit-learn's `min_samples_leaf`, which constrains the size of both children. The comparison keeps this setting at its default value of `1` and focuses on the shared high-level algorithm.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier

from decision_tree import DecisionTreeClassifier

RANDOM_STATE = 42
MAX_DEPTH = 3
iris = load_iris()
X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
    iris.data,
    iris.target,
    np.arange(len(iris.target)),
    test_size=0.2,
    stratify=iris.target,
    random_state=RANDOM_STATE,
)

print(f"Training shape: {X_train.shape}; test shape: {X_test.shape}")
print(f"Class names: {iris.target_names.tolist()}")

## One controlled fit

Both models are fit once with `max_depth=3`. The table reports independently observable outcomes. Accuracy measures correctness against the labels; agreement measures whether the two models make the same prediction for each test row.

In [ ]:
custom = DecisionTreeClassifier(min_leaf_size=1, max_depth=MAX_DEPTH)
custom.fit(X_train, y_train, max_depth=MAX_DEPTH)

sklearn_tree = SklearnDecisionTreeClassifier(
    criterion="gini",
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
sklearn_tree.fit(X_train, y_train)

custom_pred = custom.predict(X_test).astype(int)
sklearn_pred = sklearn_tree.predict(X_test).astype(int)
same_prediction = custom_pred == sklearn_pred

single_fit = pd.DataFrame(
    {
        "model": ["custom", "scikit-learn"],
        "test_accuracy": [
            accuracy_score(y_test, custom_pred),
            accuracy_score(y_test, sklearn_pred),
        ],
        "tree_depth": [custom.root.depth(), sklearn_tree.get_depth()],
        "node_count": [custom.root.count_nodes(), sklearn_tree.tree_.node_count],
    }
)
display(single_fit)
print(
    f"Exact prediction agreement: {same_prediction.mean():.1%} "
    f"({same_prediction.sum()}/{len(same_prediction)} test rows)"
)

## Depth sweep: behavior across multiple tree sizes

A single holdout result can be accidental. This sweep repeats the controlled fit for depths 1 through 5 while keeping the data split fixed. It makes changes in accuracy, agreement, and structural complexity visible instead of selecting one favorable result.

In [ ]:
sweep_rows = []
for depth in range(1, 6):
    custom_model = DecisionTreeClassifier(min_leaf_size=1, max_depth=depth)
    custom_model.fit(X_train, y_train, max_depth=depth)
    sklearn_model = SklearnDecisionTreeClassifier(
        criterion="gini", max_depth=depth, random_state=RANDOM_STATE
    )
    sklearn_model.fit(X_train, y_train)

    custom_depth_pred = custom_model.predict(X_test).astype(int)
    sklearn_depth_pred = sklearn_model.predict(X_test).astype(int)
    sweep_rows.append(
        {
            "max_depth": depth,
            "custom_accuracy": accuracy_score(y_test, custom_depth_pred),
            "sklearn_accuracy": accuracy_score(y_test, sklearn_depth_pred),
            "agreement": np.mean(custom_depth_pred == sklearn_depth_pred),
            "custom_depth": custom_model.root.depth(),
            "sklearn_depth": sklearn_model.get_depth(),
            "custom_nodes": custom_model.root.count_nodes(),
            "sklearn_nodes": sklearn_model.tree_.node_count,
        }
    )

depth_sweep = pd.DataFrame(sweep_rows)
display(depth_sweep.round({"custom_accuracy": 3, "sklearn_accuracy": 3, "agreement": 3}))

## Error analysis for the controlled fit

The confusion matrices use the same class order. The disagreement table retains the original Iris row index, true class, and both predictions so that differences are inspectable rather than hidden behind a single accuracy score.

In [ ]:
matrices = [
    (confusion_matrix(y_test, custom_pred, labels=range(3)), "Custom tree"),
    (confusion_matrix(y_test, sklearn_pred, labels=range(3)), "Scikit-learn"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (matrix, title) in zip(axes, matrices):
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(3), iris.target_names)
    ax.set_yticks(range(3), iris.target_names)
    for (row, column), value in np.ndenumerate(matrix):
        ax.text(column, row, value, ha="center", va="center")
fig.colorbar(image, ax=axes, shrink=0.8, label="count")
plt.tight_layout()
plt.show()

disagreement = pd.DataFrame(
    {
        "iris_row": test_indices[~same_prediction],
        "true_class": iris.target_names[y_test[~same_prediction]],
        "custom_prediction": iris.target_names[custom_pred[~same_prediction]],
        "sklearn_prediction": iris.target_names[sklearn_pred[~same_prediction]],
    }
)
display(disagreement)
print(f"Different predictions: {len(disagreement)} of {len(y_test)} test rows")

## What the comparison does and does not show

The two models share the same greedy idea: search numeric feature thresholds, choose a split that improves weighted Gini impurity, and stop when a depth, purity, or split-validity condition is reached. Close accuracy and substantial prediction agreement are useful evidence that the core implementation behaves sensibly on this dataset.

Exact structural or prediction equality is not expected. The custom implementation has different split tie-breaking, breadth-first growth, stopping semantics, and probability output conventions. Scikit-learn also provides production-level input validation, optimized split search, richer stopping controls, and mature edge-case handling. Differences in this notebook should therefore be interpreted as diagnostic evidence, not automatically as bugs or as proof of superiority.